In [ ]:
# =============================================================================
# Experiment 8: Email Spoofing Pattern Detection
# Objective: Using an email dataset, detect patterns of email spoofing. [CO3]
# =============================================================================

# 1. Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# =============================================================================
# 2. Create / Load Synthetic Email Header & Metadata Dataset
# Simulating key email authentication headers (From, Return-Path, Reply-To, SPF, DKIM, DMARC)
# Label: 0 = Legitimate Email, 1 = Spoofed Email
# =============================================================================

email_records = [
    # Legitimate emails (aligned domains, passing SPF/DKIM/DMARC)
    {'from_addr': 'security@github.com', 'return_path': 'bounce@github.com', 'reply_to': 'noreply@github.com',
     'spf': 'pass', 'dkim': 'pass', 'dmarc': 'pass', 'received_hops': 3, 'urgent_subject': 0, 'label': 0},
    {'from_addr': 'alerts@hdfcbank.net', 'return_path': 'mailer@hdfcbank.net', 'reply_to': 'support@hdfcbank.net',
     'spf': 'pass', 'dkim': 'pass', 'dmarc': 'pass', 'received_hops': 4, 'urgent_subject': 0, 'label': 0},
    {'from_addr': 'admissions@jainuniversity.ac.in', 'return_path': 'info@jainuniversity.ac.in', 'reply_to': 'admissions@jainuniversity.ac.in',
     'spf': 'pass', 'dkim': 'pass', 'dmarc': 'pass', 'received_hops': 2, 'urgent_subject': 0, 'label': 0},
    {'from_addr': 'no-reply@accounts.google.com', 'return_path': 'gaia@accounts.google.com', 'reply_to': 'no-reply@accounts.google.com',
     'spf': 'pass', 'dkim': 'pass', 'dmarc': 'pass', 'received_hops': 3, 'urgent_subject': 1, 'label': 0},
    {'from_addr': 'orders@amazon.in', 'return_path': 'bounces@amazon.in', 'reply_to': 'orders@amazon.in',
     'spf': 'pass', 'dkim': 'pass', 'dmarc': 'pass', 'received_hops': 4, 'urgent_subject': 0, 'label': 0},
    {'from_addr': 'billing@cloudflare.com', 'return_path': 'billing@cloudflare.com', 'reply_to': 'billing@cloudflare.com',
     'spf': 'pass', 'dkim': 'pass', 'dmarc': 'pass', 'received_hops': 3, 'urgent_subject': 0, 'label': 0},
    {'from_addr': 'notifications@linkedin.com', 'return_path': 'bounce@linkedin.com', 'reply_to': 'notifications@linkedin.com',
     'spf': 'pass', 'dkim': 'pass', 'dmarc': 'pass', 'received_hops': 3, 'urgent_subject': 0, 'label': 0},
    {'from_addr': 'team@coursera.org', 'return_path': 'mailer@coursera.org', 'reply_to': 'team@coursera.org',
     'spf': 'pass', 'dkim': 'pass', 'dmarc': 'pass', 'received_hops': 3, 'urgent_subject': 0, 'label': 0},
    
    # Spoofed emails (domain mismatches, lookalike domains, failed SPF/DKIM/DMARC)
    {'from_addr': 'security@paypal.com', 'return_path': 'attacker@mail-relay-99.xyz', 'reply_to': 'collect@phish-drop.ru',
     'spf': 'fail', 'dkim': 'none', 'dmarc': 'fail', 'received_hops': 7, 'urgent_subject': 1, 'label': 1},
    {'from_addr': 'admin@micros0ft-support.com', 'return_path': 'bot@unknown-vps.tk', 'reply_to': 'admin@micros0ft-support.com',
     'spf': 'softfail', 'dkim': 'fail', 'dmarc': 'fail', 'received_hops': 8, 'urgent_subject': 1, 'label': 1},
    {'from_addr': 'kyc-update@sbi.co.in', 'return_path': 'promo@bulk-sender-12.info', 'reply_to': 'helpdesk@sbi-verify.top',
     'spf': 'fail', 'dkim': 'fail', 'dmarc': 'fail', 'received_hops': 6, 'urgent_subject': 1, 'label': 1},
    {'from_addr': 'ceo@company-corp.com', 'return_path': 'ext-user@freemail-proxy.biz', 'reply_to': 'wire-transfer@external-acct.com',
     'spf': 'fail', 'dkim': 'none', 'dmarc': 'fail', 'received_hops': 7, 'urgent_subject': 1, 'label': 1},
    {'from_addr': 'support@app1e.com', 'return_path': 'noreply@cheap-host-88.ml', 'reply_to': 'verify@app1e-login.cf',
     'spf': 'none', 'dkim': 'fail', 'dmarc': 'fail', 'received_hops': 9, 'urgent_subject': 1, 'label': 1},
    {'from_addr': 'hr@jainuniversity.ac.in', 'return_path': 'mailer@spoofed-server.gq', 'reply_to': 'hr-docs@gmail.com',
     'spf': 'fail', 'dkim': 'none', 'dmarc': 'fail', 'received_hops': 6, 'urgent_subject': 1, 'label': 1},
    {'from_addr': 'service@amaz0n-billing.net', 'return_path': 'bounce@amaz0n-billing.net', 'reply_to': 'refund@claim-now.xyz',
     'spf': 'softfail', 'dkim': 'none', 'dmarc': 'fail', 'received_hops': 7, 'urgent_subject': 1, 'label': 1},
    {'from_addr': 'alert@icicibank.com', 'return_path': 'sender@compromised-node.pw', 'reply_to': 'alert@icici-otp.top',
     'spf': 'fail', 'dkim': 'fail', 'dmarc': 'fail', 'received_hops': 8, 'urgent_subject': 1, 'label': 1}
] * 40

df = pd.DataFrame(email_records).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Total Emails: {len(df)} | Legitimate (0): {(df['label']==0).sum()} | Spoofed (1): {(df['label']==1).sum()}")
print(df.head())

In [ ]:
# =============================================================================
# 3. Extract Email Spoofing & Header Alignment Indicators
# Checking From vs. Return-Path domain alignment, Reply-To mismatch, typosquatting, and auth checks.
# =============================================================================

def get_domain(email_addr):
    return email_addr.split('@')[-1].lower().strip()

def extract_spoofing_features(df):
    feat = pd.DataFrame()
    
    from_domains = df['from_addr'].apply(get_domain)
    return_domains = df['return_path'].apply(get_domain)
    reply_domains = df['reply_to'].apply(get_domain)
    
    # Domain alignment features (1 = mismatch detected, 0 = aligned)
    feat['from_return_mismatch'] = (from_domains != return_domains).astype(int)
    feat['from_reply_mismatch'] = (from_domains != reply_domains).astype(int)
    
    # Lookalike / Typosquatting indicator (digits substituting letters like '0' or '1' in domain)
    feat['has_homoglyph_digits'] = from_domains.apply(lambda d: 1 if re.search(r'[a-z][01][a-z]', d) else 0)
    
    # Authentication protocol results encoded numerically (pass=1, softfail/none/fail=0)
    feat['spf_pass'] = (df['spf'] == 'pass').astype(int)
    feat['dkim_pass'] = (df['dkim'] == 'pass').astype(int)
    feat['dmarc_pass'] = (df['dmarc'] == 'pass').astype(int)
    
    # Routing & Behavioral indicators
    feat['received_hops'] = df['received_hops']
    feat['urgent_subject'] = df['urgent_subject']
    
    return feat

X = extract_spoofing_features(df)
y = df['label']

print("--- Extracted Spoofing Detection Features ---")
print(X.head())

In [ ]:
# =============================================================================
# 4. Train Classifier to Detect Spoofed Email Patterns
# =============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f"Spoofing Detection Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Spoofed']))

In [ ]:
# =============================================================================
# 5. Visualize Key Email Spoofing Indicators & Confusion Matrix
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=['Legitimate', 'Spoofed'], yticklabels=['Legitimate', 'Spoofed'], ax=axes[0])
axes[0].set_title('Email Spoofing Detection Confusion Matrix')
axes[0].set_xlabel('Predicted Status')
axes[0].set_ylabel('Actual Status')

# Feature Importance
importances = pd.Series(clf.feature_importances_, index=X.columns).sort_values()
importances.plot(kind='barh', color='firebrick', ax=axes[1])
axes[1].set_title('Top Indicators of Email Spoofing')
axes[1].set_xlabel('Feature Importance Score')

plt.tight_layout()
plt.show()